In [1]:
import os, sys

DATA_PATH = '../data/processed_v2.csv'


SRC_PATH = os.path.abspath('../src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'DATA_PATH: {DATA_PATH}')
print(f'SRC_PATH : {SRC_PATH}')

DATA_PATH: ../data/processed_v2.csv
SRC_PATH : /Users/anastasiiakostyrka/Desktop/labs/src


In [2]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv(DATA_PATH)
print(f'Рядків у processed_v2: {len(df_raw)}')

def extract_clean(text):
    text = str(text).strip()
    if text and text[-1] in ['0', '1']:
        return text[:-1].strip()
    return text.strip()

df_raw['text_clean'] = df_raw['text_v2'].apply(extract_clean)

before = len(df_raw)
df = df_raw[df_raw['text_clean'].str.split().str.len() >= 5].copy().reset_index(drop=True)
after = len(df)

print(f'До фільтрації:  {before} документів')
print(f'Після фільтрації (>=5 слів): {after} документів')
print(f'Відфільтровано: {before - after} коротких')
print(f'Середня довжина (слів): {df["text_clean"].str.split().str.len().mean():.1f}')

texts = df['text_clean']
print('\nКілька прикладів:')
for t in texts.sample(5, random_state=42):
    print(' -', t[:120])

Рядків у processed_v2: 9522
До фільтрації:  9522 документів
Після фільтрації (>=5 слів): 9509 документів
Відфільтровано: 13 коротких
Середня довжина (слів): 12.5

Кілька прикладів:
 - гідравлічні гальма shimano працюють чітко і плавно
 - напрацьовувалося забагато запущених ділянок і пустирів з бурянами та руїнами
 - у музичних школах учні мають можливість навчатися грі на різних інструментах та вокалу
 - якість зйомки в цієї камери вища
 - недоступне обладнання та надто високі вимоги щодо фототехніки для прослуховування базових курсів


In [3]:
from topic_modeling import build_tfidf_vectorizer, build_count_vectorizer

# TF-IDF для LSA
vec_tfidf = build_tfidf_vectorizer(ngram_range=(1,2), min_df=5, max_df=0.80)
X_tfidf = vec_tfidf.fit_transform(texts)

# CountVectorizer для LDA
vec_count = build_count_vectorizer(ngram_range=(1,1), min_df=5, max_df=0.80)
X_count = vec_count.fit_transform(texts)

print(f'TF-IDF матриця:  {X_tfidf.shape}  (docs x features)')
print(f'Count матриця:   {X_count.shape}')
print(f'\nTF-IDF параметри: ngram=(1,2), min_df=5, max_df=0.80')
print(f'Count  параметри: ngram=(1,1), min_df=5, max_df=0.80')
print(f'\nПриклад TF-IDF токенів (перші 20):')
print(list(vec_tfidf.get_feature_names_out()[:20]))

TF-IDF матриця:  (9509, 4841)  (docs x features)
Count матриця:   (9509, 3845)

TF-IDF параметри: ngram=(1,2), min_df=5, max_df=0.80
Count  параметри: ngram=(1,1), min_df=5, max_df=0.80

Приклад TF-IDF токенів (перші 20):
['10', '15', '15 хвилин', '30', '35055', '3dпринтерів', '45', '9001', 'band', 'bose', 'bose quietcomfort', 'c1', 'dance', 'delonghi', 'delonghi ecam', 'ecam', 'ecam 35055', 'eventагентства', 'f2j6ws0w', 'g14']


In [4]:
from topic_modeling import fit_lsa, topics_summary_df

# LSA k=5
vec_lsa5, svd5, dt_lsa5, fn_lsa5 = fit_lsa(texts, n_topics=5, ngram_range=(1,2),
                                              min_df=5, max_df=0.80)
print('=== LSA k=5 ===')
print(topics_summary_df(svd5, fn_lsa5, n_top=10).to_string(index=False))

print()

# LSA k=8
vec_lsa8, svd8, dt_lsa8, fn_lsa8 = fit_lsa(texts, n_topics=8, ngram_range=(1,2),
                                              min_df=5, max_df=0.80)
print('=== LSA k=8 ===')
print(topics_summary_df(svd8, fn_lsa8, n_top=10).to_string(index=False))

=== LSA k=5 ===
 topic                                                                                                                          top_words
     0     ціни, послуги, якості, ціни послуги, відповідають якості, відповідають, завищені, роботи, якості роботи, завищені відповідають
     1 відгуки, позитивні, позитивні відгуки, негативні, негативні відгуки, роботу, відгуки роботу, відгуки послуги, відгуки курси, курси
     2                  пропонують, послуг, широкий, спектр, спектр послуг, ремонту, вибір, пропонують широкий, обмежений, широкий спектр
     3                          місті, пропонують, послугами, обладнання, ремонту, спектр, спектр послуг, послуг, використовують, широкий
     4                                                   були, багато, ремонту, завдяки, місті, була, занадто, швидко, хочу, використання

=== LSA k=8 ===
 topic                                                                                                                          top_words
 

In [5]:
from topic_modeling import fit_lda

# LDA k=5
vec_lda5, lda5, dt_lda5, fn_lda5 = fit_lda(texts, n_topics=5, ngram_range=(1,1),
                                              min_df=5, max_df=0.80)
print('=== LDA k=5 ===')
print(topics_summary_df(lda5, fn_lda5, n_top=10).to_string(index=False))

print()

# LDA k=8
vec_lda8, lda8, dt_lda8, fn_lda8 = fit_lda(texts, n_topics=8, ngram_range=(1,1),
                                              min_df=5, max_df=0.80)
print('=== LDA k=8 ===')
print(topics_summary_df(lda8, fn_lda8, n_top=10).to_string(index=False))

=== LDA k=5 ===
 topic                                                                                           top_words
     0        ремонту, відгуки, обладнання, рівень, ціни, якість, була, обслуговування, завищені, вартість
     1 роботи, пропонують, занадто, використання, персонал, встановлення, сервісу, себе, послуг, обмежений
     2           послуги, ціни, якості, відповідають, компанії, позитивні, має, місті, проблеми, послугами
     3                             завдяки, вибір, щодо, були, тепер, швидко, можу, сервіс, широкий, життя
     4             можливість, іноді, процес, знайти, різних, легко, замовлення, зручний, здоровя, приємно

=== LDA k=8 ===
 topic                                                                                           top_words
     0            ціни, якості, обладнання, послуг, відповідають, якість, була, процес, завищені, вартість
     1           роботи, занадто, персонал, харчування, сервісу, міста, багатьох, місця, допомогти, нашого
    

In [6]:
from topic_utils import print_topics

LSA8_NAMES = [
    'Ціни та якість послуг',
    'Відгуки та навчання',
    'Сервіс у місті / звернення',
    'Асортимент та вибір послуг',
    'Обладнання та ремонт',
    'Доставка та замовлення',
    'Технічний шум / стоп-слова',  # погана тема
    'Дублікат: обладнання/ремонт', # погана тема
]

LDA8_NAMES = [
    'Ціни/якість/ремонт',
    'Ресурси та впровадження',
    'Навчання та здоровʼя',
    'Сервіс та атмосфера',
    'Асортимент послуг',
    'Доставка та замовлення',
    'Загальна оцінка / змішана',   
    'Додаток / цифровий сервіс',
]

print('======== LSA k=8 ========')
print_topics(svd8, fn_lsa8, topic_names=LSA8_NAMES, n_top=10)

print('\n\n======== LDA k=8 ========')
print_topics(lda8, fn_lda8, topic_names=LDA8_NAMES, n_top=10)

======== LSA k=8 ========

Topic  0 | Ціни та якість послуг
  Words: ціни, послуги, якості, ціни послуги, відповідають якості, відповідають, завищені, роботи, якості роботи, завищені відповідають

Topic  1 | Відгуки та навчання
  Words: відгуки, позитивні, позитивні відгуки, негативні, негативні відгуки, роботу, відгуки роботу, відгуки послуги, відгуки курси, курси

Topic  2 | Сервіс у місті / звернення
  Words: пропонують, послуг, широкий, спектр, спектр послуг, ремонту, вибір, пропонують широкий, обмежений, широкий спектр

Topic  3 | Асортимент та вибір послуг
  Words: місті, послугами, пропонують, спектр, обладнання, ремонту, послуг, спектр послуг, широкий, використовують

Topic  4 | Обладнання та ремонт
  Words: багато, були, ремонту, завдяки, була, занадто, місті, постійно, людей, персонал

Topic  5 | Доставка та замовлення
  Words: обладнання, місті, використовують, послугами, сучасне, сучасне обладнання, запчастини, ремонту, застаріле, користувався

Topic  6 | Технічний шум / ст

In [7]:
from topic_utils import print_top_docs

print('=== LDA k=8: Top documents per topic ===\n')
for i, name in enumerate(LDA8_NAMES):
    print_top_docs('LDA k=8', i, name, dt_lda8, texts, n_docs=2)

print('\n=== LSA k=8: Top documents для 3 найкращих тем ===\n')
for i in [0, 2, 4]:
    print_top_docs('LSA k=8', i, LSA8_NAMES[i], dt_lsa8, texts, n_docs=2)

=== LDA k=8: Top documents per topic ===


--- LDA k=8 | Topic 0: Ціни/якість/ремонт | Top docs ---
  [doc_id=3045, score=0.9125]
  ціни на навчання у школах іноземних мов завищені та не відповідають якості викладання...

  [doc_id=2803, score=0.9028]
  ціни у ресторанах та кафе завищені та не відповідають якості страв та обслуговування...


--- LDA k=8 | Topic 1: Ресурси та впровадження | Top docs ---
  [doc_id=125, score=0.8906]
  мої діти роблять мене щасливим кожного дня | я пишаюся їхніми досягненнями і радію спостерігати...

  [doc_id=1641, score=0.8905]
  пацієнтів ніхто не поважає та не дбає про їхній комфорт | обслуговування абсолютно неякісне та безвідповідальне...


--- LDA k=8 | Topic 2: Навчання та здоровʼя | Top docs ---
  [doc_id=603, score=0.9125]
  з самого початку дотримуються графіка робіт та узгоджених кошторисів | обіцянки справді виконують у термін...

  [doc_id=491, score=0.9125]
  є зручні функції пошуку та вибору інструментів за характеристиками | сайт інтуїтив

In [8]:
import pandas as pd

interpretations = [
    {
        'model': 'LDA k=8',
        'topic': 0,
        'name': 'Ціни, якість та ремонт',
        'quality': 'GOOD',
        'comment': (
            'Тема охоплює відгуки про відповідність цін якості послуг і ремонту. '
            'Ключові слова: ціни, якості, послуги, ремонту, відповідають, завищені. '
            'Документи підтверджують: тексти скарг на завищені ціни та повільний ремонт. '
            'Тема чітка і корисна для моніторингу якості сервісу.'
        )
    },
    {
        'model': 'LDA k=8',
        'topic': 2,
        'name': 'Навчання та здоровʼя',
        'quality': 'GOOD',
        'comment': (
            'Тема групує звернення про освіту, можливості навчання та здоровʼя. '
            'Слова: навчання, можливості, здоровʼя, дітей, нових. '
            'Документи: відгуки про школи, курси, медичні послуги. '
            'Тема реальна і відповідає доменові корпусу.'
        )
    },
    {
        'model': 'LDA k=8',
        'topic': 3,
        'name': 'Сервіс та атмосфера',
        'quality': 'GOOD',
        'comment': (
            'Тема про загальний рівень сервісу: персонал, харчування, атмосфера. '
            'Слово "завдяки" вказує на позитивні відгуки. '
            'Документи: ресторани, готелі, заклади обслуговування. '
            'Тема добре читається і не змішує різні домени.'
        )
    },
    {
        'model': 'LDA k=8',
        'topic': 7,
        'name': 'Додаток / цифровий сервіс',
        'quality': 'MODERATE',
        'comment': (
            'Тема фокусується на цифрових сервісах: додаток, зручний, сервіс, знайти. '
            'Відносно вузька, але реальна. Частково змішана з темою доставки. '
            'Корисна для аналізу UX-відгуків.'
        )
    },
    {
        'model': 'LDA k=8',
        'topic': 6,
        'name': 'Загальна оцінка / змішана',
        'quality': 'BAD',
        'comment': (
            'Тема надто загальна: вартість, дизайн, організатори, люди. '
            'Не прив\'язана до одного домену — змішані сюжети. '
            'Документи різнорідні: від скарг на вартість до коментарів про дизайн. '
            'Це типова "сміттєва корзина" теми — погана.'
        )
    },
    {
        'model': 'LSA k=8',
        'topic': 6,
        'name': 'Технічний шум / стоп-слова',
        'quality': 'BAD',
        'comment': (
            'Тема побудована навколо залишкових стоп-слів і шаблонних конструкцій. '
            'LSA уловив стиль тексту, а не зміст. '
            'Документи не мають спільної тематики. '
            'Причина: недостатнє фільтрування частих службових слів.'
        )
    },
    {
        'model': 'LSA k=8',
        'topic': 7,
        'name': 'Дублікат: обладнання/ремонт',
        'quality': 'BAD',
        'comment': (
            'Тема майже повністю дублює тему 4 (обладнання та ремонт). '
            'Топ-слова: обладнання, ремонту, використовують, сучасне — ті самі, що в T4. '
            'LSA при k=8 "розщепив" одну реальну тему на дві. '
            'Рішення: зменшити k до 6 або переглянути min_df.'
        )
    },
]

df_interp = pd.DataFrame(interpretations)
print(df_interp[['model','topic','name','quality']].to_string(index=False))

print('\n--- Детальні пояснення ---')
for row in interpretations:
    print(f"\n[{row['model']} | T{row['topic']} | {row['quality']}] {row['name']}")
    print(f"  {row['comment']}")

  model  topic                        name  quality
LDA k=8      0      Ціни, якість та ремонт     GOOD
LDA k=8      2        Навчання та здоровʼя     GOOD
LDA k=8      3         Сервіс та атмосфера     GOOD
LDA k=8      7   Додаток / цифровий сервіс MODERATE
LDA k=8      6   Загальна оцінка / змішана      BAD
LSA k=8      6  Технічний шум / стоп-слова      BAD
LSA k=8      7 Дублікат: обладнання/ремонт      BAD

--- Детальні пояснення ---

[LDA k=8 | T0 | GOOD] Ціни, якість та ремонт
  Тема охоплює відгуки про відповідність цін якості послуг і ремонту. Ключові слова: ціни, якості, послуги, ремонту, відповідають, завищені. Документи підтверджують: тексти скарг на завищені ціни та повільний ремонт. Тема чітка і корисна для моніторингу якості сервісу.

[LDA k=8 | T2 | GOOD] Навчання та здоровʼя
  Тема групує звернення про освіту, можливості навчання та здоровʼя. Слова: навчання, можливості, здоровʼя, дітей, нових. Документи: відгуки про школи, курси, медичні послуги. Тема реальна і відпо

In [9]:
print('=== АНАЛІЗ ПОГАНИХ ТЕМ ===')

bad_topics = [
    {
        'model': 'LDA k=8, Topic 6',
        'name': 'Загальна оцінка / змішана',
        'problem': 'mixed topic',
        'why': (
            'Корпус неоднорідний за доменами (готелі, освіта, ремонт, ІТ). '
            'При k=8 LDA "з\'їдає" залишок у одну змішану тему. '
            'Слова вартість/дизайн/організатори не мають спільного сюжету.'
        ),
        'fix': 'Збільшити k до 10–12 або розбити корпус на домени перед моделюванням.',
    },
    {
        'model': 'LSA k=8, Topic 6',
        'name': 'Технічний шум / стоп-слова',
        'problem': 'stop-word / template topic',
        'why': (
            'LSA уловив статистичний патерн шаблонної лексики. '
            'TruncatedSVD не має вбудованої тематичної «розрядності» — '
            'перша компонента іноді поглинає службові конструкції. '
            'Стоп-словний список не охопив усі шаблонні фрази корпусу.'
        ),
        'fix': 'Розширити UA_STOP_WORDS, додати max_df=0.70, перевірити на lemma_text.',
    },
    {
        'model': 'LSA k=8, Topic 7',
        'name': 'Дублікат теми обладнання/ремонт',
        'problem': 'duplicate topic',
        'why': (
            'LSA при великому k "розщеплює" одну реальну тему на кілька компонент. '
            'Topic 4 і Topic 7 мають понад 60% спільних топ-слів. '
            'Характерна проблема SVD — він шукає ортогональні вектори, '
            'але не гарантує тематичної різноманітності.'
        ),
        'fix': 'Зменшити k до 5–6 для LSA, або перейти на LDA де softmax нормалізація '
               'краще розподіляє вагу.'
    },
]

for bt in bad_topics:
    print(f"\n{'='*55}")
    print(f"Модель/Тема: {bt['model']} — '{bt['name']}'")
    print(f"Тип проблеми: {bt['problem']}")
    print(f"Чому погана: {bt['why']}")
    print(f"Що фіксити: {bt['fix']}")

=== АНАЛІЗ ПОГАНИХ ТЕМ ===

Модель/Тема: LDA k=8, Topic 6 — 'Загальна оцінка / змішана'
Тип проблеми: mixed topic
Чому погана: Корпус неоднорідний за доменами (готелі, освіта, ремонт, ІТ). При k=8 LDA "з'їдає" залишок у одну змішану тему. Слова вартість/дизайн/організатори не мають спільного сюжету.
Що фіксити: Збільшити k до 10–12 або розбити корпус на домени перед моделюванням.

Модель/Тема: LSA k=8, Topic 6 — 'Технічний шум / стоп-слова'
Тип проблеми: stop-word / template topic
Чому погана: LSA уловив статистичний патерн шаблонної лексики. TruncatedSVD не має вбудованої тематичної «розрядності» — перша компонента іноді поглинає службові конструкції. Стоп-словний список не охопив усі шаблонні фрази корпусу.
Що фіксити: Розширити UA_STOP_WORDS, додати max_df=0.70, перевірити на lemma_text.

Модель/Тема: LSA k=8, Topic 7 — 'Дублікат теми обладнання/ремонт'
Тип проблеми: duplicate topic
Чому погана: LSA при великому k "розщеплює" одну реальну тему на кілька компонент. Topic 4 і Topic 7 

In [10]:
import matplotlib.pyplot as plt
from topic_utils import compare_models_table
from topic_modeling import get_top_words

lsa_topics = get_top_words(svd8, fn_lsa8, n_top=8)
lda_topics = get_top_words(lda8, fn_lda8, n_top=8)

cmp_df = compare_models_table(lsa_topics, lda_topics,
                               lsa_names=LSA8_NAMES,
                               lda_names=LDA8_NAMES)
print('=== LSA k=8 vs LDA k=8 ===')
print(cmp_df.to_string(index=False))

print('''
=== Висновок: яка модель краща для цього корпусу ===

LDA (k=8) дала більш читабельні теми для цього корпусу з кількох причин:

1. Тематична чистота: LDA явно моделює розподіл слів PER тему через dirichlet prior,
   що дає менш "розмиті" теми порівняно з SVD-компонентами LSA.

2. Дублікати: LSA при k=8 розщепила тему 'обладнання/ремонт' на дві майже ідентичні
   (T4 та T7). LDA цього не зробила — теми більш диверсифіковані.

3. Інтерпретованість: слова в темах LDA мають вищу семантичну зв'язність —
   наприклад, тема 'сервіс та атмосфера' (персонал, харчування, атмосфера)
   легко читається, тоді як відповідна LSA-тема містить бігами без чіткого фокусу.

4. Шум: LSA Topic 6 — типова 'стоп-словна' тема, яка у LDA не виникає так явно,
   бо LDA нормалізує ваги через softmax.

5. Де LSA краща: швидше тренується, краще підходить для retrieval-задач
   (косинусна подібність документів), але для ручної інтерпретації
   у цьому корпусі LDA виграє.

Рекомендація: використовувати LDA k=8 як основу для подальшого аналізу,
LSA залишити для порівняння та retrieval-експериментів.
''')

=== LSA k=8 vs LDA k=8 ===
                  LSA назва                                                                                                LSA top words                 LDA назва                                                               LDA top words
      Ціни та якість послуг                     ціни, послуги, якості, ціни послуги, відповідають якості, відповідають, завищені, роботи        Ціни/якість/ремонт        ціни, якості, обладнання, послуг, відповідають, якість, була, процес
        Відгуки та навчання відгуки, позитивні, позитивні відгуки, негативні, негативні відгуки, роботу, відгуки роботу, відгуки послуги   Ресурси та впровадження      роботи, занадто, персонал, харчування, сервісу, міста, багатьох, місця
 Сервіс у місті / звернення                       пропонують, послуг, широкий, спектр, спектр послуг, ремонту, вибір, пропонують широкий      Навчання та здоровʼя    послуги, вибір, місті, компанії, широкий, послугами, приємно, пропонують
 Асортимент та ви

In [11]:
import os

summary = f"""# Audit Summary Lab 8 — Topic Modeling LSA / LDA

## 1. Розмір корпусу після фільтрації
- До фільтрації: {len(df_raw)} документів
- Після фільтрації (>=5 слів): {len(df)} документів
- Середня довжина: {df['text_clean'].str.split().str.len().mean():.1f} слів

## 2. Моделі, що протестовано
- LSA (TF-IDF + TruncatedSVD)
- LDA (CountVectorizer + LatentDirichletAllocation)

## 3. Значення k, що перевірялись
- k=5, k=8 для обох моделей

## 4. Найкращі теми (LDA k=8)
1. Topic 0: Ціни, якість та ремонт — чітка, підтверджена документами
2. Topic 2: Навчання та здоровʼя — реальний домен, добре відокремлена
3. Topic 3: Сервіс та атмосфера — позитивні відгуки, читабельна

## 5. Найгірші теми
1. LDA T6: Змішана загальна тема — різнорідний контент, 'сміттєва корзина'
2. LSA T6: Стоп-словна тема — уловлено стиль, а не зміст
3. LSA T7: Дублікат теми обладнання/ремонт від LSA при k=8

## 6. Причини поганих тем
- Неоднорідний корпус (змішані домени: готелі, освіта, ІТ, ремонт)
- Неповний список стоп-слів
- LSA SVD "розщеплює" теми при великому k

## 7. Яка модель краща для цього кейсу
LDA k=8 — більш читабельні та менш дубльовані теми.
LSA корисна для retrieval, але гірше інтерпретується вручну.

## 8. Що робити далі
- Спробувати lemma_text замість processed_v2
- Розширити список стоп-слів
- Спробувати k=10 для LDA
- Опційно: перевірити coherence score для вибору k
"""

out_path = '../docs/audit_summary_lab8.md'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(summary)

print(f'Звіт збережено: {out_path}')
print(summary)

Звіт збережено: ../docs/audit_summary_lab8.md
# Audit Summary Lab 8 — Topic Modeling LSA / LDA

## 1. Розмір корпусу після фільтрації
- До фільтрації: 9522 документів
- Після фільтрації (>=5 слів): 9509 документів
- Середня довжина: 12.5 слів

## 2. Моделі, що протестовано
- LSA (TF-IDF + TruncatedSVD)
- LDA (CountVectorizer + LatentDirichletAllocation)

## 3. Значення k, що перевірялись
- k=5, k=8 для обох моделей

## 4. Найкращі теми (LDA k=8)
1. Topic 0: Ціни, якість та ремонт — чітка, підтверджена документами
2. Topic 2: Навчання та здоровʼя — реальний домен, добре відокремлена
3. Topic 3: Сервіс та атмосфера — позитивні відгуки, читабельна

## 5. Найгірші теми
1. LDA T6: Змішана загальна тема — різнорідний контент, 'сміттєва корзина'
2. LSA T6: Стоп-словна тема — уловлено стиль, а не зміст
3. LSA T7: Дублікат теми обладнання/ремонт від LSA при k=8

## 6. Причини поганих тем
- Неоднорідний корпус (змішані домени: готелі, освіта, ІТ, ремонт)
- Неповний список стоп-слів
- LSA SVD "ро